# ReviewIQ – Data Preprocessing

## Objective

Prepare the raw customer review dataset for
Natural Language Processing and Machine Learning.

## Preprocessing Steps

- Handle missing values
- Remove unnecessary columns
- Generate sentiment labels
- Combine review title and text
- Clean textual data
- Save the processed datase
t

In [1]:
import pandas as pd
import numpy as np
import re
import os
import html
import nltk

from pathlib import Path
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
DATA_PATH = Path("../data/raw")

csv_files = list(DATA_PATH.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Dataset not found!")

df = pd.read_csv(csv_files[0])

print("Dataset loaded:", df.shape)

Dataset loaded: (23486, 11)


In [3]:
reviews_df = df.copy()

print("Working dataset:", reviews_df.shape)

Working dataset: (23486, 11)


# Remove Unnecessary Columns

In [4]:
reviews_df.drop(
    columns=['Unnamed: 0'],
    errors='ignore',
    inplace=True
)
print("Remaining columns:", reviews_df.columns.tolist())

Remaining columns: ['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']


# Handle Missing Values

In [5]:
print("Before handling missing values:", reviews_df.shape)

reviews_df = reviews_df.dropna(
    subset=['Review Text']
).copy()

print("After removing missing review text:", reviews_df.shape)

Before handling missing values: (23486, 10)
After removing missing review text: (22641, 10)


In [6]:
reviews_df['Title'] = reviews_df['Title'].fillna('')

In [7]:
category_columns = [
    'Division Name',
    'Department Name',
    'Class Name'
]

reviews_df[category_columns] = reviews_df[
    category_columns
].fillna('Unknown')

In [8]:
print(reviews_df.isnull().sum())

Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64


# Create Sentiment Labels

In [9]:
def assign_sentiment(rating):

    if rating <= 2:
        return 'Negative'

    elif rating == 3:
        return 'Neutral'

    else:
        return 'Positive'


reviews_df['Sentiment'] = reviews_df['Rating'].apply(
    assign_sentiment
)

In [10]:
print(reviews_df[['Rating', 'Sentiment']].head(10))

print("\nSentiment Distribution:")
print(reviews_df['Sentiment'].value_counts())

   Rating Sentiment
0       4  Positive
1       5  Positive
2       3   Neutral
3       5  Positive
4       5  Positive
5       2  Negative
6       5  Positive
7       4  Positive
8       5  Positive
9       5  Positive

Sentiment Distribution:
Sentiment
Positive    17448
Neutral      2823
Negative     2370
Name: count, dtype: int64


In [11]:
reviews_df['Combined_Review'] = (
    reviews_df['Title'].str.strip() + ' ' +
    reviews_df['Review Text'].str.strip()
).str.strip()

reviews_df[
    ['Title', 'Review Text', 'Combined_Review']
].head()

,Title,Review Text,Combined_Review
0,,Absolutely wonderful - silky and sexy and comf...,Absolutely wonderful - silky and sexy and comf...
1,,Love this dress! it's sooo pretty. i happene...,Love this dress! it's sooo pretty. i happene...
2,Some major design flaws,I had such high hopes for this dress and reall...,Some major design flaws I had such high hopes ...
3,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...","My favorite buy! I love, love, love this jumps..."
4,Flattering shirt,This shirt is very flattering to all due to th...,Flattering shirt This shirt is very flattering...


## 8. Text Cleaning

Customer reviews contain:
- Uppercase and lowercase letters
- Punctuation
- HTML tags
- URLs
- Extra whitespaces
- Special characters

We need to clean the text before converting it
into numerical features for machine learning.

In [12]:
for review in reviews_df['Combined_Review'].head(5):
    print(review)
    print("-" * 80)

Absolutely wonderful - silky and sexy and comfortable
--------------------------------------------------------------------------------
Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.
--------------------------------------------------------------------------------
Some major design flaws I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net ove

In [13]:
# Download required NLTK resources
nltk_resources = [
    'punkt',
    'punkt_tab',
    'stopwords',
    'wordnet',
    'omw-1.4'
]

for resource in nltk_resources:
    nltk.download(resource, quiet=True)

print("NLTK resources are ready!")

NLTK resources are ready!


In [14]:
def clean_text(text):

    # Convert to string
    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Convert to lowercase
    text = text.lower()

    # Expand common negative contractions
    contractions = {
        "don't": "do not",
        "doesn't": "does not",
        "didn't": "did not",
        "isn't": "is not",
        "aren't": "are not",
        "wasn't": "was not",
        "weren't": "were not",
        "haven't": "have not",
        "hasn't": "has not",
        "hadn't": "had not",
        "won't": "will not",
        "wouldn't": "would not",
        "couldn't": "could not",
        "shouldn't": "should not",
        "can't": "cannot"
    }

    for contraction, expanded in contractions.items():
        text = re.sub(
            re.escape(contraction),
            expanded,
            text
        ) 
        
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    text = re.sub(r"(?<=\w)['’](?=\w)", "", text)

    # Remove special characters except apostrophes
    text = re.sub(r"[^a-zA-Z\s']", ' ', text)

    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [15]:
reviews_df['Cleaned_Review'] = reviews_df[
    'Combined_Review'
].apply(clean_text)

reviews_df[
    ['Combined_Review', 'Cleaned_Review']
].head(10)

,Combined_Review,Cleaned_Review
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky and sexy and comfor...
1,Love this dress! it's sooo pretty. i happene...,love this dress its sooo pretty i happened to ...
2,Some major design flaws I had such high hopes ...,some major design flaws i had such high hopes ...
3,"My favorite buy! I love, love, love this jumps...",my favorite buy i love love love this jumpsuit...
4,Flattering shirt This shirt is very flattering...,flattering shirt this shirt is very flattering...
5,Not for the very petite I love tracy reese dre...,not for the very petite i love tracy reese dre...
6,Cagrcoal shimmer fun I aded this in my basket ...,cagrcoal shimmer fun i aded this in my basket ...
7,"Shimmer, surprisingly goes with lots I ordered...",shimmer surprisingly goes with lots i ordered ...
8,Flattering I love this dress. i usually get an...,flattering i love this dress i usually get an ...
9,"Such a fun dress! I'm 5""5' and 125 lbs. i orde...",such a fun dress im ' and lbs i ordered the s ...


# Tokenization

In [16]:
reviews_df['Tokens'] = reviews_df[
    'Cleaned_Review'
].apply(word_tokenize)

reviews_df[['Cleaned_Review', 'Tokens']].head()

,Cleaned_Review,Tokens
0,absolutely wonderful silky and sexy and comfor...,"[absolutely, wonderful, silky, and, sexy, and,..."
1,love this dress its sooo pretty i happened to ...,"[love, this, dress, its, sooo, pretty, i, happ..."
2,some major design flaws i had such high hopes ...,"[some, major, design, flaws, i, had, such, hig..."
3,my favorite buy i love love love this jumpsuit...,"[my, favorite, buy, i, love, love, love, this,..."
4,flattering shirt this shirt is very flattering...,"[flattering, shirt, this, shirt, is, very, fla..."


# Stopword Removal

In [17]:
stop_words = set(stopwords.words('english'))

# Preserve important negation words
negation_words = {'no', 'not', 'nor', "don't", "doesn't",
                  "didn't", "isn't", "wasn't", "weren't",
                  "won't", "wouldn't", "couldn't", "shouldn't"}

stop_words = stop_words - negation_words

In [18]:
reviews_df['Filtered_Tokens'] = reviews_df['Tokens'].apply(
    lambda tokens: [
        word for word in tokens
        if word not in stop_words
        and word.isalpha()
    ]
)

reviews_df[
    ['Tokens', 'Filtered_Tokens']
].head()

,Tokens,Filtered_Tokens
0,"[absolutely, wonderful, silky, and, sexy, and,...","[absolutely, wonderful, silky, sexy, comfortable]"
1,"[love, this, dress, its, sooo, pretty, i, happ...","[love, dress, sooo, pretty, happened, find, st..."
2,"[some, major, design, flaws, i, had, such, hig...","[major, design, flaws, high, hopes, dress, rea..."
3,"[my, favorite, buy, i, love, love, love, this,...","[favorite, buy, love, love, love, jumpsuit, fu..."
4,"[flattering, shirt, this, shirt, is, very, fla...","[flattering, shirt, shirt, flattering, due, ad..."


# Lemmatization

In [19]:
lemmatizer = WordNetLemmatizer()

reviews_df['Processed_Review'] = reviews_df[
    'Filtered_Tokens'
].apply(
    lambda tokens: ' '.join(
        lemmatizer.lemmatize(word)
        for word in tokens
    )
)

In [20]:
sample = reviews_df.iloc[0]

print("Original Review:")
print(sample['Combined_Review'])

print("\nCleaned Review:")
print(sample['Cleaned_Review'])

print("\nTokens:")
print(sample['Tokens'])

print("\nAfter Stopword Removal:")
print(sample['Filtered_Tokens'])

print("\nFinal Processed Review:")
print(sample['Processed_Review'])

Original Review:
Absolutely wonderful - silky and sexy and comfortable

Cleaned Review:
absolutely wonderful silky and sexy and comfortable

Tokens:
['absolutely', 'wonderful', 'silky', 'and', 'sexy', 'and', 'comfortable']

After Stopword Removal:
['absolutely', 'wonderful', 'silky', 'sexy', 'comfortable']

Final Processed Review:
absolutely wonderful silky sexy comfortable


In [21]:
test_reviews = [
    "This dress is not comfortable",
    "This product is very good",
    "I don't like this material",
    "The quality is never good"
]

for review in test_reviews:

    cleaned = clean_text(review)
    tokens = word_tokenize(cleaned)

    filtered = [
        word for word in tokens
        if word not in stop_words
        and word.isalpha()
]

    processed = [
        lemmatizer.lemmatize(word)
        for word in filtered
    ]

    print("Original:", review)
    print("Processed:", processed)
    print("-" * 60)

Original: This dress is not comfortable
Processed: ['dress', 'not', 'comfortable']
------------------------------------------------------------
Original: This product is very good
Processed: ['product', 'good']
------------------------------------------------------------
Original: I don't like this material
Processed: ['not', 'like', 'material']
------------------------------------------------------------
Original: The quality is never good
Processed: ['quality', 'never', 'good']
------------------------------------------------------------


In [22]:
empty_reviews = (
    reviews_df['Processed_Review'].str.strip() == ''
)

print("Empty processed reviews:", empty_reviews.sum())

Empty processed reviews: 0


In [23]:
reviews_df['Original_Length'] = (
    reviews_df['Combined_Review'].str.split().str.len()
)

reviews_df['Processed_Length'] = (
    reviews_df['Processed_Review'].str.split().str.len()
)

print("Original average words:",
      reviews_df['Original_Length'].mean())

print("Processed average words:",
      reviews_df['Processed_Length'].mean())

Original average words: 63.103131487125125
Processed average words: 31.660615697186522


In [24]:
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

columns_to_save = [
    'Clothing ID',
    'Age',
    'Title',
    'Review Text',
    'Rating',
    'Recommended IND',
    'Positive Feedback Count',
    'Division Name',
    'Department Name',
    'Class Name',
    'Sentiment',
    'Combined_Review',
    'Cleaned_Review',
    'Processed_Review'
]

processed_df = reviews_df[columns_to_save].copy()

processed_df.to_csv(
    PROCESSED_PATH / "cleaned_reviews.csv",
    index=False
)

print("Processed dataset saved successfully!")
print("Shape:", processed_df.shape)

Processed dataset saved successfully!
Shape: (22641, 14)
